In [1]:
import pandas as pd

In [2]:
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv()

class PostgresSQL():
    def __init__(self):
        self.host = "localhost"
        self.port = 5432
        self.database = "data_source"
        self.user = os.getenv("POSTGRES_USER")
        self.password = os.getenv("POSTGRES_PASSWORD")
 
    def connect(self):
        connection = psycopg2.connect(
            host=self.host,
            port=self.port,
            database=self.database,
            user=self.user,
            password=self.password
        )
        return connection
    
    def cursor(self):
        return self.connect().cursor()
    
    def close(self):
        self.connect().close()


In [3]:
p = PostgresSQL()
conn = p.connect()
c = conn.cursor()

In [7]:
c.execute("SELECT * FROM transactions WHERE amount < 0 ORDER BY transaction_time DESC LIMIT 10")
pd.DataFrame(c.fetchall(), columns=[desc[0] for desc in c.description])

,transaction_id,customer_id,merchant_id,amount,currency,status,payment_method,transaction_time,updated_at
0,aa41cb55-0b5a-4fb7-a1b9-e3f728178e94,62,9,-469656.99,VND,SUCCESS,CASH,2026-04-15 14:12:11.566349,2026-04-15 14:12:11.566349
1,055daf12-a005-4e85-a216-39715282ce6b,10,3,-392288.75,VND,SUCCESS,CASH,2026-04-15 14:12:09.225715,2026-04-15 14:12:09.225715
2,b825aa5e-15c4-417b-939b-c24f3d19a835,17,2,-252049.86,VND,SUCCESS,CASH,2026-04-15 14:11:09.190678,2026-04-15 14:11:09.190678
3,adfa22e1-73ec-4819-a8fb-d9ad1f163295,93,10,-410177.65,VND,SUCCESS,CASH,2026-04-15 14:11:04.121649,2026-04-15 14:11:04.121649
4,17e5d9bf-2781-4cf0-a5da-f11068d28748,11,11,-384460.73,VND,SUCCESS,CASH,2026-04-15 14:10:49.276745,2026-04-15 14:10:49.276745
5,14781b49-505e-4876-a0e9-4bf0b7e880ed,88,9,-488004.28,VND,SUCCESS,CASH,2026-04-15 14:10:35.479047,2026-04-15 14:10:35.479047
6,38dcd4fa-6616-45f1-acdf-f871875d40eb,30,4,-53707.16,VND,SUCCESS,CASH,2026-04-15 14:10:26.473996,2026-04-15 14:10:26.473996


In [5]:
c.execute("SELECT COUNT(*) FROM customers")
c.fetchone()

(185,)

In [ ]:
p.close()

In [1]:
test = []
test.append(1)
test.pop()

1

In [3]:
i = 2
for i in range(5):
    print(i)

0
1
2
3
4


### Test code clickhouse

In [8]:
import clickhouse_connect
client = clickhouse_connect.get_client(host='localhost', port=8123, user='clickhouse', password='clickhouse')
print(client.command("SHOW DATABASES"))

INFORMATION_SCHEMA
default
information_schema
streaming
system


In [9]:
client.command("""
CREATE TABLE IF NOT EXISTS streaming.kafka_enriched_transactions
(
    transaction_id     String,
    customer_id        Int32,
    customer_name      String,
    customer_city      String,
    merchant_id        Int32,
    merchant_name      String,
    merchant_category  String,
    amount             Decimal(15, 2),
    currency           String,
    status             String,
    payment_method     String,
    transaction_time   DateTime64(3)
)
ENGINE = Kafka()
SETTINGS
    kafka_broker_list      = 'kafka:9092',
    kafka_topic_list       = 'processed.enriched_transactions',
    kafka_group_name       = 'clickhouse-enriched-transactions',
    kafka_format           = 'AvroConfluent',
    format_avro_schema_registry_url = 'http://schema-registry:8081',
    kafka_handle_error_mode = 'stream';
""")

In [12]:
client.command("""
CREATE TABLE IF NOT EXISTS streaming.enriched_transactions
(
    transaction_id     String,
    customer_id        Int32,
    customer_name      String,
    customer_city      String,
    merchant_id        Int32,
    merchant_name      String,
    merchant_category  String,
    amount             Decimal(15, 2),
    currency           String,
    status             String,
    payment_method     String,
    transaction_time   DateTime64(3),
    ingested_at        DateTime DEFAULT now()
)
ENGINE = ReplacingMergeTree(transaction_time) -- If have any duplicate transaction_id, keep the one with the latest transaction_time
PARTITION BY toYYYYMM(transaction_time)
ORDER BY transaction_id;

""")

In [13]:
client.command("""
CREATE MATERIALIZED VIEW IF NOT EXISTS streaming.mv_enriched_transactions 
TO streaming.enriched_transactions AS
SELECT
    transaction_id,
    customer_id,
    customer_name,
    customer_city,
    merchant_id,
    merchant_name,
    merchant_category,
    amount,
    currency,
    status,
    payment_method,
    transaction_time
FROM streaming.kafka_enriched_transactions;
""")

In [15]:
print(client.command("SHOW TABLES FROM streaming"))

enriched_transactions
kafka_enriched_transactions
mv_enriched_transactions


In [17]:
client.query_df("SELECT * FROM streaming.enriched_transactions LIMIT 10")

,transaction_id,customer_id,customer_name,customer_city,merchant_id,merchant_name,merchant_category,amount,currency,status,payment_method,transaction_time,ingested_at
0,,0,,,0,,,0.00,,,,1970-01-01,2026-05-02 10:28:26
